# End-to-end pipeline demo

One pass through every agent, in the order they run in production:

1. **Factor Researcher** — reads papers, brainstorms features, codes them up, IC-tests them.
2. **Selector** — picks a hypothesis + the factors that should encode it.
3. **Architect** — combines the selected factors into a strategy; iterates with a backtest loop.
4. **Statistician** — OOS validation, deflated Sharpe, approval gate → writes to `strategy_db`.
5. **Portfolio Manager** — allocates capital across the approved strategies (with optional multi-PM committee).

Each stage has a toggle at the top so you can skip the slow LLM calls and demo whichever parts you want.

In [ ]:
# ── Demo configuration ─────────────────────────────────────────
# Set any of these to False to skip the LLM call for that stage
# (uses whatever is already persisted in the DBs).
RUN_FACTOR_RESEARCH = False    # heaviest stage — minutes per session
RUN_SELECTOR_AND_ARCHITECT = True
RUN_STATISTICIAN = True
RUN_PORTFOLIO_MANAGER = True
USE_COMMITTEE = True            # if True, runs 3 PMs and aggregates

import logging, os, time
from dotenv import load_dotenv
load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-7s  %(name)-22s  %(message)s", datefmt="%H:%M:%S")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in your .env before running.")

print("Pipeline config:")
for k, v in [("factor research", RUN_FACTOR_RESEARCH), ("selector+architect", RUN_SELECTOR_AND_ARCHITECT),
             ("statistician", RUN_STATISTICIAN), ("portfolio manager", RUN_PORTFOLIO_MANAGER),
             ("  → committee mode", USE_COMMITTEE)]:
    print(f"  {k:20s} = {v}")

## 1 · Factor Researcher

Reads one paper from `data/papers/`, brainstorms factor ideas, generates Python
implementations, runs a quick IC backtest, and persists anything passing the
threshold into `data/factors/factor_db.json` (with code under
`quant_fund_agent/factors/researcher/`).

In [ ]:
from quant_fund_agent.factors import discover_factors
discover_factors()

if RUN_FACTOR_RESEARCH:
    from quant_fund_agent.agents.factor_research.graph import factor_research_graph
    from quant_fund_agent.agents.factor_research.state import FactorResearcherState

    t0 = time.perf_counter()
    research = factor_research_graph.invoke(FactorResearcherState(
        session_id="demo",
        n_papers=1, n_factor_ideas=2,
        ic_threshold=0.01, ic_target_horizon=6,
        n_tickers=10,
    ))
    print(f"\n>>> Factor research finished in {time.perf_counter() - t0:.1f}s")
    print(f"   Papers read     : {len(research.get('selected_papers', []))}")
    print(f"   Ideas generated : {len(research.get('factor_ideas', []))}")
    print(f"   Kept            : {research.get('kept_factor_ids', [])}")
    print(f"   Rejected        : {research.get('rejected_factor_ids', [])}")
else:
    print("(skipped — using existing factor_db.json)")
    import json
    with open("data/factors/factor_db.json") as f:
        fdb = json.load(f)
    print(f"  factor_db.json contains {len(fdb.get('factors', []))} factors")

## 2 · Selector + Architect

* **Selector** picks a hypothesis (e.g. *"momentum + microstructure short-term reversal"*)
  and which factors should encode it.
* **Architect** combines those factors into a strategy spec, runs a backtest, refines
  the weights/horizon, and decides `approve | reject | escalate`.

In [ ]:
selector_result = architect_result = None

if RUN_SELECTOR_AND_ARCHITECT:
    from quant_fund_agent.agents.selector.graph import selector_graph
    from quant_fund_agent.agents.architect.graph import architect_graph

    # ── Selector ──
    t0 = time.perf_counter()
    selector_result = selector_graph.invoke({})
    print(f">>> Selector finished in {time.perf_counter() - t0:.1f}s")
    print(f"   Hypothesis : {selector_result['hypothesis']}")
    print(f"   Factors    : {selector_result['selected_factor_ids']}")

    # ── Architect ──
    t0 = time.perf_counter()
    architect_result = architect_graph.invoke({
        "hypothesis": selector_result["hypothesis"],
        "selected_factor_ids": selector_result["selected_factor_ids"],
        "factor_catalog": selector_result["factor_catalog"],
        "max_iterations": 3,
        "oos_split_ratio": 0.2,
    })
    print(f"\n>>> Architect finished in {time.perf_counter() - t0:.1f}s")
    print(f"   Trials run : {len(architect_result['trial_history'])}")
    for t in architect_result["trial_history"]:
        m = t.metrics
        print(f"     · trial {t.iteration}: Sharpe={m.get('sharpe_ratio')}  MaxDD={m.get('max_drawdown')}")
    print(f"\n   Decision   : {architect_result['decision'].upper()}")
    print(f"   Reason     : {architect_result['decision_reasoning'][:200]}…")
else:
    print("(skipped)")

## 3 · Statistician

OOS evaluation: deflated Sharpe, walk-forward stability, bootstrap CIs.  Approved
strategies are written to `data/strategies/strategy_db.json` and become available
to the Portfolio Manager.

In [ ]:
stat_result = None

if RUN_STATISTICIAN and architect_result is not None and architect_result["decision"] == "approve":
    from quant_fund_agent.agents.statistician.graph import statistician_graph

    t0 = time.perf_counter()
    stat_result = statistician_graph.invoke({
        "hypothesis":          architect_result["hypothesis"],
        "strategy_spec":       architect_result["strategy_spec"],
        "trial_history":       architect_result["trial_history"],
        "backtest_metrics":    architect_result["backtest_metrics"],
        "initial_reasoning":   architect_result.get("initial_reasoning", ""),
        "final_reasoning":     architect_result["strategy_spec"].reasoning,
        "oos_split_ratio":     0.2,
    })
    print(f">>> Statistician finished in {time.perf_counter() - t0:.1f}s\n")
    for r in stat_result["test_results"]:
        status = "PASS" if r.passed else "FAIL"
        print(f"  [{status}] {r.test_name:32s}  {r.metric_value}")
    print(f"\n   Final verdict : {stat_result['final_decision'].upper()}")
elif RUN_STATISTICIAN:
    print("(architect did not approve — statistician skipped)")
else:
    print("(skipped)")

## 4 · Portfolio Manager

Loads every approved strategy from `strategy_db.json` and produces an allocation.
Two modes shown:

* **Single PM** — one personality + one construction method.
* **Committee** — three PMs (defensive / balanced / aggressive) each propose,
  the committee aggregates into one consensus `PortfolioRecord`.

In [ ]:
if not RUN_PORTFOLIO_MANAGER:
    print("(skipped)")
else:
    from pathlib import Path
    from quant_fund_agent.agents.portfolio_manager.graph import portfolio_manager_graph
    from quant_fund_agent.agents.portfolio_manager.state import PortfolioManagerState
    from quant_fund_agent.agents.portfolio_manager.committee import (
        CommitteeConfig, run_pm_committee,
    )
    from quant_fund_agent.databases import PortfolioDatabase, StrategyDatabase
    from quant_fund_agent.portfolio import get_personality_profile
    from quant_fund_agent.schemas import PMMode, PMPersonality, VotingMethod

    STRATEGY_DB_PATH = Path("data/strategies/strategy_db.json")
    PORTFOLIO_DB_PATH = Path("data/portfolio/portfolio_db.json")

    sdb = StrategyDatabase()
    if STRATEGY_DB_PATH.exists():
        sdb.load_from_json(STRATEGY_DB_PATH)
    pdb = PortfolioDatabase()
    if PORTFOLIO_DB_PATH.exists():
        pdb.load_from_json(PORTFOLIO_DB_PATH)

    print(f"Strategies in book: {len(sdb.list_strategies())}")
    if len(sdb.list_strategies()) == 0:
        print("⚠  No strategies persisted yet — run the upstream agents first "
              "(or enable RUN_SELECTOR_AND_ARCHITECT + RUN_STATISTICIAN above).")
    else:
        def make_pm(personality, name=None):
            return PortfolioManagerState(
                pm_name=name or f"pm_{personality.value}",
                mode=PMMode.SELECTOR,
                personality=personality,
                profile=get_personality_profile(personality),
                strategy_db=sdb,
                portfolio_db=pdb,
            )

        if USE_COMMITTEE:
            print("\n— Running 3-PM committee (SIMPLE_AVERAGE consensus) —")
            record = run_pm_committee(
                [make_pm(p) for p in (PMPersonality.DEFENSIVE,
                                       PMPersonality.BALANCED,
                                       PMPersonality.AGGRESSIVE)],
                CommitteeConfig(voting_method=VotingMethod.SIMPLE_AVERAGE,
                                pm_name="demo_committee"),
                sdb, pdb,
            )
        else:
            print("\n— Running single balanced PM —")
            result = portfolio_manager_graph.invoke(make_pm(PMPersonality.BALANCED, "demo_pm"))
            record = result["portfolio_record"]

        print(f"\nPortfolio:           {record.id}")
        print(f"PM name / mode:      {record.pm_name} / {record.mode.value}")
        print(f"Construction method: {record.construction_method.value}")
        print(f"# allocations:       {len([a for a in record.allocations if a.enabled])}")
        print(f"Flagged:             {record.flagged_strategy_ids}")
        print(f"Retired:             {record.retired_strategy_ids}")
        print("\nWeights:")
        for a in record.allocations:
            if a.enabled:
                print(f"  {a.strategy_id:<35s}  w = {a.weight:+.4f}")
        if record.expected_metrics:
            print("\nExpected portfolio metrics:")
            for k, v in record.expected_metrics.items():
                print(f"  {k:30s}  {v}")

        STRATEGY_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
        PORTFOLIO_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
        sdb.save_to_json(STRATEGY_DB_PATH)
        pdb.save_to_json(PORTFOLIO_DB_PATH)
        print("\nDBs persisted ✓")

---

**That's the whole pipeline.**  Each agent is independently testable
(see `tests.ipynb` for unit-level coverage) and the runners
(`run_factor_research.py`, `run_selector.py`, `run_architect.py`,
`run_statistician.py`, `run_portfolio_manager.py`) wrap the same
graphs for CLI use.